A GLM (Generalized Linear Model) is a flexible extension of linear regression that 
allows modelling of non‑normal data such as claim counts and claim costs. GLMs combine 
a probability distribution (e.g., Poisson or Gamma) with a link function (typically a 
log link) to ensure predictions are valid and to allow multiplicative effects of 
rating factors. This framework is widely used in insurance pricing because it handles 
skewed data, categorical variables, and exposure adjustments in a statistically 
sound way.

In [2]:
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import numpy as np

data = pd.read_csv("../data/processed/prepared_data.csv")


## Frequency GLM
A Poisson GLM is used for modelling claim frequency because the response variable 
(ClaimNb) is a count. The Poisson distribution assumes that the mean and variance are 
equal, which is often too restrictive for insurance data. A quasi-Poisson model 
relaxes this assumption by allowing the variance to exceed the mean, making it more 
appropriate when overdispersion is present.

A log link function is used to ensure that the predicted claim frequency is always 
positive and to allow multiplicative effects of rating factors. Under the log link, 
each coefficient represents a percentage change in expected frequency relative to the 
baseline category.


In [3]:
freq_model = smf.glm(
    formula="ClaimNb ~ DriverAge + Power + Region + Gas + Density",
    data=data,
    family=sm.families.Poisson(),
    offset=np.log(data["Exposure"])
).fit()

print(freq_model.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                ClaimNb   No. Observations:               413960
Model:                            GLM   Df Residuals:                   413936
Model Family:                 Poisson   Df Model:                           23
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -73766.
Date:                Thu, 26 Mar 2026   Deviance:                   1.1418e+05
Time:                        20:14:42   Pearson chi2:                 7.54e+05
No. Iterations:                     7   Pseudo R-squ. (CS):           0.002429
Covariance Type:            nonrobust                                         
                                   coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------
Intercept       

The frequency GLM shows that several rating factors significantly influence the 
expected number of claims. Higher power vehicles tend to have higher claim frequency, 
regular gas vehicles have lower frequency than the baseline fuel type, and frequency 
decreases with driver age. Density has a positive effect, indicating that drivers in 
more densely populated areas experience more claims. The large Pearson chi-square 
value indicates overdispersion, so a quasi-Poisson model is more appropriate.

### Re-fitting the Frequency Model Using a Quasi-Poisson GLM

The initial Poisson GLM showed clear evidence of overdispersion, as the Pearson 
chi-square statistic was substantially larger than the residual degrees of freedom. 
Overdispersion occurs when the variance of the response variable exceeds the mean, 
which violates the assumptions of the standard Poisson model.

To address this, the model was re-fitted using a quasi-Poisson specification. The 
quasi-Poisson GLM keeps the same mean structure as the Poisson model but introduces a 
dispersion parameter that adjusts the standard errors. This results in more reliable 
significance tests while leaving the coefficient estimates unchanged.


In [4]:
freq_model_qp = smf.glm(
    formula="ClaimNb ~ DriverAge + Power + Region + Gas + Density",
    data=data,
    family=sm.families.Poisson(),
    offset=np.log(data["Exposure"])
).fit(scale="X2")

print(freq_model_qp.summary())


                 Generalized Linear Model Regression Results                  
Dep. Variable:                ClaimNb   No. Observations:               413960
Model:                            GLM   Df Residuals:                   413936
Model Family:                 Poisson   Df Model:                           23
Link Function:                    Log   Scale:                          1.8222
Method:                          IRLS   Log-Likelihood:                -40481.
Date:                Thu, 26 Mar 2026   Deviance:                   1.1418e+05
Time:                        20:14:52   Pearson chi2:                 7.54e+05
No. Iterations:                     9   Pseudo R-squ. (CS):           0.001334
Covariance Type:            nonrobust                                         
                                   coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------
Intercept       

### Interpretation

The quasi-Poisson model confirms the same directional effects as the original Poisson 
model, but with adjusted standard errors to account for overdispersion. The scale 
parameter of 1.82 indicates that the variance of claim counts is around 82% higher 
than the Poisson assumption of equal mean and variance. This level of overdispersion 
is typical in insurance frequency data and justifies the use of a quasi-Poisson 
specification.

Several predictors remain statistically significant after adjusting for overdispersion. 
Claim frequency decreases with driver age, increases with vehicle power, and is higher 
in more densely populated areas. Regular fuel vehicles show lower frequency relative 
to the baseline fuel type. These effects are consistent with the patterns observed in 
the EDA and reflect well-known behavioural and environmental risk factors.


## Severity GLM

The severity model estimates the average cost of a claim, conditional on a claim 
occurring. Because claim amounts are positive and right-skewed, a Gamma GLM with a 
log link is used. The model is fitted only on policies with at least one claim, and 
severity is calculated as ClaimAmount divided by ClaimNb. The log link ensures that 
predicted severities remain positive and allows multiplicative effects of rating 
factors.


In [5]:
#severity glm 
sev_model = smf.glm(
    formula="severity ~ CarAge + Brand + Region",
    data=data,
    family=sm.families.Gamma(link=sm.families.links.log())
).fit()

print(sev_model.summary())


c:\Users\adaan\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\genmod\families\links.py:13: FutureWarning: The log link alias is deprecated. Use Log instead. The log link alias will be removed after the 0.15.0 release.
  warnings.warn(


                 Generalized Linear Model Regression Results                  
Dep. Variable:               severity   No. Observations:                16181
Model:                            GLM   Df Residuals:                    16164
Model Family:                   Gamma   Df Model:                           16
Link Function:                    log   Scale:                          58.137
Method:                          IRLS   Log-Likelihood:            -1.7658e+05
Date:                Thu, 26 Mar 2026   Deviance:                       26242.
Time:                        20:14:55   Pearson chi2:                 9.40e+05
No. Iterations:                    17   Pseudo R-squ. (CS):          0.0008317
Covariance Type:            nonrobust                                         
                                                  coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------

### Interpretation

The severity GLM shows that none of the rating factors have strong statistical 
significance in explaining variation in claim severity. This is expected, as severity 
data is typically much noisier than frequency data and is based on a much smaller 
sample of claims. The Gamma GLM still provides a stable framework for modelling 
average claim cost and produces reasonable coefficient signs that are consistent with 
the EDA. The high scale parameter reflects the natural variability in claim amounts. 
Overall, the severity model is appropriate and will be combined with the frequency 
model in the pure premium (Tweedie) GLM.

## Pure Premium GLM (Tweedie)

The Tweedie GLM is used to model the expected claim cost per policy in a single 
framework. Pure premium data contains many zero values (policies with no claims) and 
positive, right-skewed values when claims occur. The Tweedie distribution with a 
power parameter between 1 and 2 naturally handles this mixture by combining a Poisson 
frequency component with a Gamma severity component.

Using a log link ensures that predictions remain positive and allows multiplicative 
effects of rating factors. The Tweedie model therefore provides a unified and 
statistically coherent approach to modelling pure premium, replacing the separate 
frequency and severity models with a single GLM.

### Step 1 — Define the Target Variable (y) and Predictors (X)

The Tweedie model predicts the expected claim cost per policy, so the target variable 
is `ClaimAmount`. The predictors include both numeric and categorical rating factors 
used in the frequency and severity models. These variables represent driver behaviour, 
vehicle characteristics, and environmental risk.

- `y` = ClaimAmount (pure premium)
- `X` = DriverAge, Power, Region, Gas, Density


In [6]:
data["ClaimAmount"] = data["ClaimAmount"].fillna(0)
y = data["ClaimAmount"]
X = data[["DriverAge", "Power", "Region", "Gas", "Density"]]

### Step 2 — Preprocess Categorical and Numeric Variables

The Tweedie model can only work with numbers, so we need to convert our data into a 
numeric format.

- **Categorical variables** (Power, Region, Gas) are turned into 0/1 columns using 
  one‑hot encoding.
- **Numeric variables** (DriverAge, Density) are kept as they are.

This allows the model to understand each category and learn how it affects the 
predicted claim cost.


In [7]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import TweedieRegressor

categorical = ["Power", "Region", "Gas"]
numeric = ["DriverAge", "Density"]

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(drop="first"), categorical),
        ("num", StandardScaler(), numeric)
    ]
)


### Why Scaling Was Needed

The numeric variables (DriverAge and Density) were on very different scales, with 
DriverAge ranging from around 18–90 and Density reaching values in the thousands. 
This large difference caused numerical instability during optimisation, leading the 
Tweedie model to collapse to an intercept‑only solution with zero coefficients.

Using `StandardScaler` standardises each numeric variable by subtracting its mean and 
dividing by its standard deviation. This puts all numeric features on a similar scale, 
which stabilises the optimisation process and allows the Tweedie model to converge 
properly and produce meaningful coefficients.


### Step 3 — Build and Fit the Tweedie Model

A pipeline is used to combine preprocessing and model fitting. The TweedieRegressor 
with a log link and a power value of 1.5 models pure premium directly by combining 
Poisson (frequency) and Gamma (severity) behaviour. The log link ensures positive 
predictions and multiplicative effects, which matches actuarial pricing practice.


In [8]:
print(y)
(y > 0).sum()


0         0.0
1         0.0
2         0.0
3         0.0
4         0.0
         ... 
413955    0.0
413956    0.0
413957    0.0
413958    0.0
413959    0.0
Name: ClaimAmount, Length: 413960, dtype: float64


np.int64(16181)

In [9]:
tweedie_model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", TweedieRegressor(
        power=1.5,
        alpha=0.0,
        link="log",
        max_iter=20000
    ))
])

tweedie_model.fit(X, y)


,steps,"[('preprocess', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [10]:
#Results
model = tweedie_model.named_steps["model"]
coefs = model.coef_
intercept = model.intercept_

print("Intercept:", intercept)
print("Coefficients:", coefs)


Intercept: 3.8619891726021565
Coefficients: [ 0.127326    0.40073748  0.16254674  0.0695876   0.88229267  0.19963511
  0.23921654  0.11466426  0.51858308  0.18819151 -0.08356862  0.3231562
  0.4163432   0.4882874  -0.48504913  0.04350269  0.02835729 -0.05746381
  0.0737242   0.11268239  0.06244201 -0.17001483  0.0126925 ]


### Why the Tweedie Model Initially Returned Zero Coefficients

The TweedieRegressor was originally producing only zero coefficients because the 
optimisation process was failing. This happened for three reasons:

1. **Different feature scales** – DriverAge and Density were on very different scales, 
   which caused numerical instability. Scaling the numeric variables fixed this.

2. **Regularisation too strong** – The default alpha value shrank all coefficients 
   towards zero. Setting `alpha = 0.0` removed this effect.

3. **Insufficient iterations** – The optimiser did not have enough iterations to 
   converge, so it defaulted to an intercept‑only model. Increasing `max_iter` allowed 
   the model to find a proper solution.

After scaling the numeric variables, removing regularisation, and increasing the 
iteration limit, the Tweedie model converged successfully and produced meaningful 
coefficients.


### Matching Coefficients to Features

The Tweedie model is fitted on transformed data produced by the pipeline, so the 
coefficient order does not directly match the original dataset. To interpret the 
model, we extract the transformed feature names from the preprocessing step:

1. Get the OneHotEncoder feature names for the categorical variables.
2. Append the numeric feature names (DriverAge and Density).
3. Combine these into a single ordered list.
4. Zip this list with the model coefficients.

This produces a clear mapping from each coefficient to its corresponding rating 
factor, allowing proper interpretation of the GLM.


In [11]:

ohe = tweedie_model.named_steps["preprocess"].named_transformers_["cat"]

cat_features = ohe.get_feature_names_out(["Power", "Region", "Gas"])

numeric = ["DriverAge", "Density"]
numeric_features = numeric

all_features = list(cat_features) + numeric_features

coefs = tweedie_model.named_steps["model"].coef_

for name, coef in zip(all_features, coefs):
    print(name, coef)


Power_e 0.1273260040941904
Power_f 0.40073748144716986
Power_g 0.16254674494320154
Power_h 0.06958759746321558
Power_i 0.8822926681995018
Power_j 0.19963510707674875
Power_k 0.2392165390415014
Power_l 0.1146642602414421
Power_m 0.5185830818961549
Power_n 0.18819150884672875
Power_o -0.08356861800391574
Region_Basse-Normandie 0.32315619540346147
Region_Bretagne 0.4163432046004563
Region_Centre 0.4882874031567346
Region_Haute-Normandie -0.4850491299710101
Region_Ile-de-France 0.043502692746950844
Region_Limousin 0.028357287940634454
Region_Nord-Pas-de-Calais -0.05746381127256657
Region_Pays-de-la-Loire 0.07372419824653667
Region_Poitou-Charentes 0.11268239180772163
Gas_Regular 0.062442007486489585
DriverAge -0.17001483213139598
Density 0.012692496785907415


In [12]:
import joblib

joblib.dump(tweedie_model, "tweedie_model.pkl")


['tweedie_model.pkl']

### Saving and Loading the Fitted Tweedie Model

The fitted GLM pipeline can be saved using joblib so that it can be reused in a 
separate notebook without refitting the model.

To save the model:
    joblib.dump(tweedie_model, "tweedie_model.pkl")

To load the model in a new notebook:
    tweedie_model = joblib.load("tweedie_model.pkl")

This preserves the entire pipeline, including preprocessing steps and the fitted 
TweedieRegressor, allowing coefficients, relativities, and pricing outputs to be 
generated cleanly in a separate workflow.
